In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path
import sys

# ── Load plot style utilities ─────────────────────────────────────────────────
NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'res_visualisation' else NOTEBOOK_DIR

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from plot_style_utils import (
    apply_plot_theme,
    PALETTE,
    THEME,
    style_axis,
    save_plot_data,
)

# # ── Override theme for thesis (white background) ──────────────────────────────
# THEME.update({
#     "figure.facecolor": "white",
#     "axes.facecolor":   "white",
#     "grid.color":       "#e8e8e8",
#     "grid.alpha":       0.8,
# })

apply_plot_theme()

# ════════════════════════════════════════════════════════════
#  CHANGE THIS WHEN SWITCHING RUN
RUN = 'Fredrik_1'
# ════════════════════════════════════════════════════════════

# ── Analysis constants ────────────────────────────────────────────────────────
START_YEAR = 2011

METHODS = ["Method1_Raw", "Method2_PostMean", "Method3_ProbQ5"]

LABELS = {
    "Method1_Raw":      "Raw signal",
    "Method2_PostMean": "Posterior mean",
    "Method3_ProbQ5":   "Probabilistic Q5",
}

METHOD_COLORS = {
    "Method1_Raw":      PALETTE["blue"],
    "Method2_PostMean": PALETTE["orange"],
    "Method3_ProbQ5":   PALETTE["green"],
}

SIGNAL_COL = {
    "Method1_Raw":      "theta_obs",
    "Method2_PostMean": "theta_post_mean",
    "Method3_ProbQ5":   "p_q5",
}

VARIANT_LINESTYLE = {
    "HB":      "-",
    "HB_cap":  "--",
    "HB_down": ":",
}

print(f"Run: {RUN}")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'res_visualisation' else NOTEBOOK_DIR

RUN_DIR   = ROOT / 'results' / RUN
PORT_EVAL = RUN_DIR / 'portfolio_evaluation'
PORT_FORM = RUN_DIR / 'portfolio_formation'

# ── Find latent_prof_model with HB/HB_cap/HB_down ────────────────────────────
def find_latent_base(run_dir):
    latent = run_dir / 'latent_prof_model'
    if latent.exists():
        subdirs = [s.name for s in latent.iterdir() if s.is_dir()]
        if all(v in subdirs for v in ['HB', 'HB_cap', 'HB_down']):
            return latent
    raise FileNotFoundError(f'Could not find latent_prof_model with HB/HB_cap/HB_down in {run_dir}')

LATENT_BASE       = find_latent_base(RUN_DIR)
LATENT_DIR_HB     = LATENT_BASE / 'HB'
LATENT_DIR_HB_CAP = LATENT_BASE / 'HB_cap'
LATENT_DIR_HB_DOWN= LATENT_BASE / 'HB_down'

# ── Portfolio eval and formation paths per variant ────────────────────────────
PORT_EVAL_HB      = PORT_EVAL / 'HB'
PORT_EVAL_HB_CAP  = PORT_EVAL / 'HB_cap'
PORT_EVAL_HB_DOWN = PORT_EVAL / 'HB_down'

PORT_FORM_HB      = PORT_FORM / 'HB'
PORT_FORM_HB_CAP  = PORT_FORM / 'HB_cap'
PORT_FORM_HB_DOWN = PORT_FORM / 'HB_down'

# ── Static input ──────────────────────────────────────────────────────────────
STEP2_CSV = ROOT / 'results' / 'extraction_static' / 'prepared_step2_input.csv'

# ── Validate paths ────────────────────────────────────────────────────────────
check_paths = {
    'HB latent':         LATENT_DIR_HB,
    'HB_cap latent':     LATENT_DIR_HB_CAP,
    'HB_down latent':    LATENT_DIR_HB_DOWN,
    'Port eval HB':      PORT_EVAL_HB,
    'Port eval HB_cap':  PORT_EVAL_HB_CAP,
    'Port eval HB_down': PORT_EVAL_HB_DOWN,
    'Port form HB':      PORT_FORM_HB,
    'Port form HB_cap':  PORT_FORM_HB_CAP,
    'Port form HB_down': PORT_FORM_HB_DOWN,
    'Step2 input':       STEP2_CSV,
}

print(f'Run: {RUN}\n')
all_ok = True
for label, path in check_paths.items():
    ok = path.exists()
    all_ok = all_ok and ok
    print(f'  {"✓" if ok else "✗ NOT FOUND":12s} {label:25s} {path}')

print()
if not all_ok:
    print('⚠️  Some paths are missing — check the RUN name in cell 1')
else:
    print('✓ All paths OK — loading data...')

    # ── Latent firm-year ──────────────────────────────────────────────────────
    firm_year_hb      = pd.read_csv(LATENT_DIR_HB      / 'latent_prof_firm_year.csv', low_memory=False)
    firm_year_hb_cap  = pd.read_csv(LATENT_DIR_HB_CAP  / 'latent_prof_firm_year.csv', low_memory=False)
    firm_year_hb_down = pd.read_csv(LATENT_DIR_HB_DOWN / 'latent_prof_firm_year.csv', low_memory=False)

    firm_year_hb['run']      = 'HB'
    firm_year_hb_cap['run']  = 'HB_cap'
    firm_year_hb_down['run'] = 'HB_down'

    # ── Portfolio formation (HB as primary) ───────────────────────────────────
    port_long      = pd.read_csv(PORT_FORM_HB / 'portfolio_assignments_long.csv')
    port_wide      = pd.read_csv(PORT_FORM_HB / 'portfolio_assignments_wide.csv')
    port_formation = pd.read_csv(PORT_FORM_HB / 'portfolio_formation_summary.csv')

    # ── Holdings per variant ──────────────────────────────────────────────────
    holdings_hb      = pd.read_csv(PORT_EVAL_HB      / 'monthly_holdings.csv', parse_dates=['Date'])
    holdings_hb_cap  = pd.read_csv(PORT_EVAL_HB_CAP  / 'monthly_holdings.csv', parse_dates=['Date'])
    holdings_hb_down = pd.read_csv(PORT_EVAL_HB_DOWN / 'monthly_holdings.csv', parse_dates=['Date'])

    holdings_hb['Variant']      = 'HB'
    holdings_hb_cap['Variant']  = 'HB_cap'
    holdings_hb_down['Variant'] = 'HB_down'

    holdings_all = pd.concat(
        [holdings_hb, holdings_hb_cap, holdings_hb_down], ignore_index=True
    )

    # ── Step2 input — equity ratio filter ────────────────────────────────────
    step2 = pd.read_csv(STEP2_CSV)
    step2["BE"] = pd.to_numeric(step2["BE"], errors="coerce")
    step2["AT"] = pd.to_numeric(step2["AT"], errors="coerce")

    equity_ratio = step2[
        step2["BE"].notna() &
        step2["AT"].notna() &
        (step2["AT"] > 0) &
        (step2["BE"] > 0)
    ].copy()
    equity_ratio["EK_ratio"]      = equity_ratio["BE"] / equity_ratio["AT"]
    equity_ratio["FormationYear"] = equity_ratio["Year"] + 1

    # ── Print shapes ──────────────────────────────────────────────────────────
    print('\nLoaded shapes:')
    for name, df in [
        ('firm_year_hb',      firm_year_hb),
        ('firm_year_hb_cap',  firm_year_hb_cap),
        ('firm_year_hb_down', firm_year_hb_down),
        ('port_long',         port_long),
        ('holdings_all',      holdings_all),
        ('equity_ratio',      equity_ratio),
    ]:
        print(f'  {name:25s}: {df.shape}')

In [ ]:
# ── Cell 3: Sample overview and sector composition ────────────────────────────
df = firm_year_hb.copy()

n_firms      = df["Ticker"].nunique()
n_firm_years = len(df)
n_years      = df["FormationYear"].nunique()
per_year     = df.groupby("FormationYear")["Ticker"].nunique().reset_index()
per_year.columns = ["Formation year", "N firms"]

# ── Headline summary ──────────────────────────────────────────────────────────
summary = pd.DataFrame({
    "Metric": ["Unique firms", "Firm-years", "Formation years", "Period", "Avg firms / year"],
    "Value":  [n_firms, n_firm_years, n_years,
               f"{df['FormationYear'].min()}–{df['FormationYear'].max()}",
               f"{n_firm_years / n_years:.1f}"]
})
display(summary.style.hide(axis="index").set_properties(**{"text-align": "left"}))

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(per_year["Formation year"], per_year["N firms"],
            color=PALETTE["blue"], edgecolor="white", linewidth=0.8)
axes[0].set_title("Firms per formation year")
axes[0].set_xlabel("Formation year")
axes[0].set_ylabel("Number of firms")
axes[0].grid(True, alpha=0.3, axis="y")

sector_counts = (
    df.groupby(["FormationYear", "Sector"])["Ticker"]
    .count()
    .unstack(fill_value=0)
)
sector_counts.plot(
    kind="bar", stacked=True, ax=axes[1],
    colormap="tab20", legend=True, width=0.85
)
axes[1].set_title("Sector composition per formation year")
axes[1].set_xlabel("Formation year")
axes[1].set_ylabel("Number of firms")
axes[1].legend(fontsize=7, bbox_to_anchor=(1, 1), title="Sector")
axes[1].tick_params(axis="x", rotation=45)
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 4: Signal distributions ─────────────────────────────────────────────
df = firm_year_hb.copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Raw PROF ──────────────────────────────────────────────────────────────────
axes[0].hist(df["PROF_w"].dropna().clip(-5, 5), bins=60,
             color=PALETTE["blue"], edgecolor="white", linewidth=0.5, alpha=0.85)
axes[0].axvline(df["PROF_w"].median(), color=PALETTE["red"],
                linewidth=1.2, linestyle="--", label=f"Median: {df['PROF_w'].median():.3f}")
axes[0].set_title("PROF distribution (clipped ±5)")
axes[0].set_xlabel("PROF (raw signal)")
axes[0].set_ylabel("Count")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, axis="y")

# ── Latent quality (posterior mean) ───────────────────────────────────────────
axes[1].hist(df["theta_post_mean"].dropna().clip(-5, 5), bins=60,
             color=PALETTE["orange"], edgecolor="white", linewidth=0.5, alpha=0.85)
axes[1].axvline(df["theta_post_mean"].median(), color=PALETTE["red"],
                linewidth=1.2, linestyle="--",
                label=f"Median: {df['theta_post_mean'].median():.3f}")
axes[1].set_title("Latent quality distribution (clipped ±5)")
axes[1].set_xlabel("Posterior mean (theta)")
axes[1].set_ylabel("Count")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3, axis="y")

# ── Sigma ─────────────────────────────────────────────────────────────────────
axes[2].hist(df["sigma_acc"].dropna(), bins=60,
             color=PALETTE["green"], edgecolor="white", linewidth=0.5, alpha=0.85)
axes[2].axvline(df["sigma_acc"].median(), color=PALETTE["red"],
                linewidth=1.2, linestyle="--",
                label=f"Median: {df['sigma_acc'].median():.3f}")
axes[2].set_title("Accounting uncertainty distribution")
axes[2].set_xlabel("sigma_acc")
axes[2].set_ylabel("Count")
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3, axis="y")

# ── Descriptive stats table ───────────────────────────────────────────────────
desc = pd.DataFrame({
    "Statistic":       df["PROF_w"].describe().index,
    "PROF (raw)":      df["PROF_w"].describe().round(4).values,
    "Latent quality":  df["theta_post_mean"].describe().round(4).values,
    "sigma_acc":       df["sigma_acc"].describe().round(4).values,
})
display(desc.style.hide(axis="index").format({
    "PROF (raw)":     "{:.4f}",
    "Latent quality": "{:.4f}",
    "sigma_acc":      "{:.4f}",
}))

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 5: PROF, latent quality and sigma by sector and exchange ─────────────

df = firm_year_hb.copy()
df["Exchange"] = df["Ticker"].str.split(".").str[-1].map({
    "OL": "Oslo (OL)",
    "ST": "Stockholm (ST)",
    "HE": "Helsinki (HE)",
    "CO": "Copenhagen (CO)",
    "IC": "Iceland (IC)",
}).fillna("Other")

exchanges = sorted(df["Exchange"].unique())

def plot_exchange(data, title_suffix=""):
    sectors     = data.groupby("Sector")["PROF_w"].median().sort_values(ascending=False).index
    x           = np.arange(len(sectors))
    width       = 0.35

    prof_vals   = data.groupby("Sector")["PROF_w"].median().reindex(sectors)
    latent_vals = data.groupby("Sector")["theta_post_mean"].median().reindex(sectors)
    sigma_vals  = data.groupby("Sector")["sigma_acc"].median().reindex(sectors)

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    # ── Grouped bar: raw vs latent ────────────────────────────────────────────
    axes[0].bar(x - width/2, prof_vals.values, width,
                label="Raw PROF",
                color=PALETTE["blue"], edgecolor="white", linewidth=0.8, alpha=0.85)
    axes[0].bar(x + width/2, latent_vals.values, width,
                label="Latent quality",
                color=PALETTE["orange"], edgecolor="white", linewidth=0.8, alpha=0.85)
    axes[0].axhline(0, color="black", linewidth=0.8, linestyle="--")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(sectors, rotation=45, ha="right")
    axes[0].set_title(f"Median PROF vs latent quality — {title_suffix}")
    axes[0].set_ylabel("Median value")
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.3, axis="y")

    # ── Sigma ─────────────────────────────────────────────────────────────────
    axes[1].bar(sectors, sigma_vals.values,
                color=PALETTE["green"], edgecolor="white", linewidth=0.8)
    axes[1].set_xticklabels(sectors, rotation=45, ha="right")
    axes[1].set_title(f"Median sigma_acc — {title_suffix}")
    axes[1].set_ylabel("Median sigma_acc")
    axes[1].grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plt.show()

# ── All exchanges combined ────────────────────────────────────────────────────
plot_exchange(df, title_suffix="all exchanges")

# ── One figure per exchange ───────────────────────────────────────────────────
for exch in exchanges:
    plot_exchange(df[df["Exchange"] == exch], title_suffix=exch)

In [ ]:
# ── Cell 6: PROF raw vs posterior mean by quintile ────────────────────────────

QUINTILES = ["Q1", "Q2", "Q3", "Q4", "Q5"]

df = firm_year_hb.copy()
df["Exchange"] = df["Ticker"].str.split(".").str[-1].map({
    "OL": "Oslo (OL)",
    "ST": "Stockholm (ST)",
    "HE": "Helsinki (HE)",
    "CO": "Copenhagen (CO)",
    "IC": "Iceland (IC)",
}).fillna("Other")

# ── Merge quintile assignments ────────────────────────────────────────────────
raw_assignments = port_long[port_long["Method"] == "Method1_Raw"][
    ["Ticker", "FormationYear", "Portfolio"]
].copy()

df = df.merge(raw_assignments, on=["Ticker", "FormationYear"], how="inner")
df = df[df["Portfolio"].isin(QUINTILES)]

CLIP = 2  # clip for display only

# ── Plot 1: Raw vs posterior mean — all exchanges ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, col, label, color in [
    (axes[0], "PROF_w",          "Raw signal",      PALETTE["blue"]),
    (axes[1], "theta_post_mean", "Posterior mean",  PALETTE["orange"]),
]:
    data = [df[df["Portfolio"] == q][col].clip(-CLIP, CLIP).dropna()
            for q in QUINTILES]

    bp = ax.boxplot(data, patch_artist=True, labels=QUINTILES,
                    medianprops=dict(color="black", linewidth=1.5),
                    flierprops=dict(marker="o", markersize=2, alpha=0.3))
    for patch in bp["boxes"]:
        patch.set_facecolor(color)
        patch.set_alpha(0.6)

    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title(f"PROF by quintile — {label}")
    ax.set_xlabel("Quintile")
    ax.set_ylabel("PROF" if ax == axes[0] else "")
    ax.grid(True, alpha=0.3, axis="y")
    ax.text(0.99, 0.01, f"Clipped at ±{CLIP}",
            transform=ax.transAxes, ha="right", va="bottom",
            fontsize=9, color=PALETTE["slate"])

fig.suptitle("Effect of Bayesian shrinkage on PROF distribution by quintile",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# ── Plot 2: Same per exchange ─────────────────────────────────────────────────
exchanges = sorted(df["Exchange"].unique())

fig, axes = plt.subplots(2, len(exchanges),
                         figsize=(5 * len(exchanges), 9),
                         sharey="row")

for col_idx, exch in enumerate(exchanges):
    sub = df[df["Exchange"] == exch]

    for row_idx, (signal_col, label, color) in enumerate([
        ("PROF_w",          "Raw signal",     PALETTE["blue"]),
        ("theta_post_mean", "Posterior mean", PALETTE["orange"]),
    ]):
        data = [sub[sub["Portfolio"] == q][signal_col].clip(-CLIP, CLIP).dropna()
                for q in QUINTILES]

        bp = axes[row_idx, col_idx].boxplot(
            data, patch_artist=True, labels=QUINTILES,
            medianprops=dict(color="black", linewidth=1.5),
            flierprops=dict(marker="o", markersize=2, alpha=0.3)
        )
        for patch in bp["boxes"]:
            patch.set_facecolor(color)
            patch.set_alpha(0.6)

        axes[row_idx, col_idx].axhline(0, color="black", linewidth=0.8, linestyle="--")
        axes[row_idx, col_idx].set_title(f"{label} — {exch}")
        axes[row_idx, col_idx].set_xlabel("Quintile")
        axes[row_idx, col_idx].set_ylabel("PROF" if col_idx == 0 else "")
        axes[row_idx, col_idx].grid(True, alpha=0.3, axis="y")

fig.suptitle("Raw vs posterior mean by quintile per exchange",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 6: PROF raw vs posterior mean and sigma by quintile ──────────────────

QUINTILES = ["Q1", "Q2", "Q3", "Q4", "Q5"]
CLIP = 2

df = firm_year_hb.copy()
df["Exchange"] = df["Ticker"].str.split(".").str[-1].map({
    "OL": "Oslo (OL)",
    "ST": "Stockholm (ST)",
    "HE": "Helsinki (HE)",
    "CO": "Copenhagen (CO)",
    "IC": "Iceland (IC)",
}).fillna("Other")

raw_assignments = port_long[port_long["Method"] == "Method1_Raw"][
    ["Ticker", "FormationYear", "Portfolio"]
].copy()
df = df.merge(raw_assignments, on=["Ticker", "FormationYear"], how="inner")
df = df[df["Portfolio"].isin(QUINTILES)]

# ── Plot 1: Raw | Posterior mean | Sigma — all exchanges ──────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

for ax, col, label, color in [
    (axes[0], "PROF_w",          "Raw signal",     PALETTE["blue"]),
    (axes[1], "theta_post_mean", "Posterior mean", PALETTE["orange"]),
]:
    data = [df[df["Portfolio"] == q][col].clip(-CLIP, CLIP).dropna()
            for q in QUINTILES]
    bp = ax.boxplot(data, patch_artist=True, labels=QUINTILES,
                    medianprops=dict(color="black", linewidth=1.5),
                    flierprops=dict(marker="o", markersize=2, alpha=0.3))
    for patch in bp["boxes"]:
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title(f"PROF by quintile — {label}")
    ax.set_xlabel("Quintile")
    ax.set_ylabel("PROF" if ax == axes[0] else "")
    ax.grid(True, alpha=0.3, axis="y")
    ax.text(0.99, 0.01, f"Clipped at ±{CLIP}",
            transform=ax.transAxes, ha="right", va="bottom",
            fontsize=9, color=PALETTE["slate"])

sigma_q = df.groupby("Portfolio")["sigma_acc"].mean().reindex(QUINTILES)
axes[2].bar(QUINTILES, sigma_q.values,
            color=PALETTE["green"], edgecolor="white", linewidth=0.8)
axes[2].set_title("Average sigma_acc by quintile")
axes[2].set_xlabel("Quintile")
axes[2].set_ylabel("Avg sigma_acc")
axes[2].grid(True, alpha=0.3, axis="y")

fig.suptitle("PROF distribution and accounting uncertainty by quintile",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# ── Plot 2: Same per exchange ─────────────────────────────────────────────────
exchanges = sorted(df["Exchange"].unique())

fig, axes = plt.subplots(3, len(exchanges),
                         figsize=(5 * len(exchanges), 12),
                         sharey="row")

for col_idx, exch in enumerate(exchanges):
    sub = df[df["Exchange"] == exch]

    for row_idx, (signal_col, label, color) in enumerate([
        ("PROF_w",          "Raw signal",     PALETTE["blue"]),
        ("theta_post_mean", "Posterior mean", PALETTE["orange"]),
    ]):
        data = [sub[sub["Portfolio"] == q][signal_col].clip(-CLIP, CLIP).dropna()
                for q in QUINTILES]
        bp = axes[row_idx, col_idx].boxplot(
            data, patch_artist=True, labels=QUINTILES,
            medianprops=dict(color="black", linewidth=1.5),
            flierprops=dict(marker="o", markersize=2, alpha=0.3)
        )
        for patch in bp["boxes"]:
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
        axes[row_idx, col_idx].axhline(0, color="black", linewidth=0.8, linestyle="--")
        axes[row_idx, col_idx].set_title(f"{label} — {exch}")
        axes[row_idx, col_idx].set_xlabel("Quintile")
        axes[row_idx, col_idx].set_ylabel("PROF" if col_idx == 0 else "")
        axes[row_idx, col_idx].grid(True, alpha=0.3, axis="y")

    sigma_q = sub.groupby("Portfolio")["sigma_acc"].mean().reindex(QUINTILES)
    axes[2, col_idx].bar(QUINTILES, sigma_q.values,
                         color=PALETTE["green"], edgecolor="white", linewidth=0.8)
    axes[2, col_idx].set_title(f"sigma_acc — {exch}")
    axes[2, col_idx].set_xlabel("Quintile")
    axes[2, col_idx].set_ylabel("Avg sigma_acc" if col_idx == 0 else "")
    axes[2, col_idx].grid(True, alpha=0.3, axis="y")

fig.suptitle("PROF and sigma by quintile per exchange",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Shrinkage and Latent Quality analysis

In [ ]:
# ── Cell 7: Shrinkage — observed vs posterior quality signal ──────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, df_var, label in [
    (axes[0], firm_year_hb,     "HB"),
    (axes[1], firm_year_hb_cap, "HB_cap"),
]:
    sample = df_var.dropna(subset=["theta_obs", "theta_post_mean", "sigma_acc"])
    sample = sample[
        sample["theta_obs"].between(
            sample["theta_obs"].quantile(0.02),
            sample["theta_obs"].quantile(0.98)
        )
    ]

    sc = ax.scatter(
        sample["theta_obs"],
        sample["theta_post_mean"],
        c=sample["sigma_acc"],
        cmap="Blues",
        alpha=0.25, s=10, rasterized=True,
        vmin=sample["sigma_acc"].quantile(0.05),
        vmax=sample["sigma_acc"].quantile(0.95),
    )

    lims = [
        sample[["theta_obs", "theta_post_mean"]].min().min(),
        sample[["theta_obs", "theta_post_mean"]].max().max(),
    ]
    ax.plot(lims, lims, color="black", linewidth=1,
            linestyle="--", label="45° — no shrinkage")

    plt.colorbar(sc, ax=ax, label="sigma_acc (accrual noise)")
    ax.set_xlabel("theta_obs (raw signal)")
    ax.set_ylabel("theta_post_mean (posterior)")
    ax.set_title(f"Shrinkage: observed vs posterior — {label}")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    ax.text(0.02, 0.98,
            "Above 45°: shrunk upward\nBelow 45°: shrunk downward\nDarker = higher sigma",
            transform=ax.transAxes, va="top", fontsize=8,
            color=PALETTE["slate"])

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 8: Shrinkage magnitude vs sigma ─────────────────────────────────────
from scipy import stats

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, df_var, label in [
    (axes[0], firm_year_hb,     "HB"),
    (axes[1], firm_year_hb_cap, "HB_cap"),
]:
    sample = df_var.dropna(subset=["theta_obs", "theta_post_mean", "sigma_acc"]).copy()
    sample = sample[
        sample["theta_obs"].between(
            sample["theta_obs"].quantile(0.02),
            sample["theta_obs"].quantile(0.98)
        )
    ]
    sample["shrinkage"] = sample["theta_post_mean"] - sample["theta_obs"]

    ax.scatter(sample["sigma_acc"], sample["shrinkage"],
               alpha=0.15, s=8, color=PALETTE["blue"], rasterized=True)
    ax.axhline(0, color="black", linewidth=1, linestyle="--")

    m, b, r, p, _ = stats.linregress(sample["sigma_acc"], sample["shrinkage"])
    xs = np.linspace(
        sample["sigma_acc"].quantile(0.01),
        sample["sigma_acc"].quantile(0.99), 100
    )
    ax.plot(xs, m * xs + b,
            color=PALETTE["red"], linewidth=2,
            label=f"OLS trend  (β={m:.2f}, R²={r**2:.3f}, p={p:.4f})")

    ax.set_xlabel("sigma_acc (accrual noise)")
    ax.set_ylabel("theta_post − theta_obs (shrinkage)")
    ax.set_title(f"Higher noise → stronger shrinkage — {label}")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 9: Lambda distribution ───────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for ax, df_var, label in [
    (axes[0], firm_year_hb,     "HB"),
    (axes[1], firm_year_hb_cap, "HB_cap"),
]:
    lam = df_var["lambda_i"].dropna()

    ax.hist(lam, bins=60,
            color=PALETTE["blue"], edgecolor="white", linewidth=0.5, alpha=0.85)

    median_lam = lam.median()
    ax.axvline(median_lam, color=PALETTE["red"], linewidth=1.5,
               linestyle="--", label=f"Median: {median_lam:.3f}")

    ax.set_xlabel("lambda_i  (1 = full weight on own signal, 0 = fully pooled)")
    ax.set_ylabel("Frequency" if ax == axes[0] else "")
    ax.set_title(f"Distribution of shrinkage weights — {label}")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

# ── Descriptive stats ─────────────────────────────────────────────────────────
desc = pd.DataFrame({
    "Statistic": firm_year_hb["lambda_i"].describe().index,
    "HB":        firm_year_hb["lambda_i"].describe().round(4).values,
    "HB_cap":    firm_year_hb_cap["lambda_i"].describe().round(4).values,
})
display(desc.style.hide(axis="index").format({
    "HB":     "{:.4f}",
    "HB_cap": "{:.4f}",
}))

In [ ]:
# ── Cell 10: Year-level hyperparameters ───────────────────────────────────────

year_summary_hb = pd.read_csv(
    LATENT_DIR_HB / "latent_prof_year_summary.csv"
)
year_summary_hb_cap = pd.read_csv(
    LATENT_DIR_HB_CAP / "latent_prof_year_summary.csv"
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, label in zip(
    axes,
    ["mu_t",    "tau2_t",                    "var_obs_t"],
    ["μ_t (market quality mean)",
     "τ²_t (between-firm variance)",
     "var_obs_t (observed variance)"],
):
    for df_ys, variant, ls in [
        (year_summary_hb,     "HB",     "-"),
        (year_summary_hb_cap, "HB_cap", "--"),
    ]:
        ax.plot(df_ys["FormationYear"], df_ys[col],
                marker="o", markersize=4,
                color=PALETTE["blue"],
                linestyle=ls, linewidth=2,
                label=variant)

    for crisis_year, crisis_label in [(2020, "2020"), (2023, "2023")]:
        ax.axvline(crisis_year, color=PALETTE["red"],
                   linewidth=1, linestyle=":",
                   label=crisis_label if ax == axes[2] else None)

    ax.set_title(label)
    ax.set_xlabel("Formation year")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

plt.suptitle("Year-level hyperparameters — HB vs HB_cap",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Portfolio Analysis

In [ ]:
# ── Cell 11: Portfolio analysis configuration ─────────────────────────────────

def assign_sector_quintiles(holdings_df, signal_col):
    pieces = []
    for (fy, sector, method), sub in holdings_df.groupby(
        ["FormationYear", "Sector", "Method"]
    ):
        unique_firms = sub.drop_duplicates("Ticker").copy()
        if len(unique_firms) < 5:
            continue
        unique_firms["NewPortfolio"] = "Q" + pd.qcut(
            unique_firms[signal_col].rank(method="first"),
            q=5, labels=[1, 2, 3, 4, 5]
        ).astype(int).astype(str)
        pieces.append(unique_firms[["Ticker", "FormationYear", "Method", "NewPortfolio"]])

    if not pieces:
        return pd.DataFrame(columns=["Ticker", "FormationYear", "Method", "NewPortfolio"])
    return pd.concat(pieces, ignore_index=True)


def compute_monthly_returns(holdings_df, weighting, industry_neutral):
    df = holdings_df.copy()

    if industry_neutral and "Sector" in df.columns:
        signal_map = {
            "Method1_Raw":      "theta_obs",
            "Method2_PostMean": "theta_post_mean",
            "Method3_ProbQ5":   "p_q5",
        }
        new_assign_pieces = []
        for method, signal_col in signal_map.items():
            if signal_col not in df.columns:
                continue
            method_df = df[df["Method"] == method].copy()
            new_assign = assign_sector_quintiles(method_df, signal_col)
            new_assign_pieces.append(new_assign)

        if new_assign_pieces:
            new_assignments = pd.concat(new_assign_pieces, ignore_index=True)
            df = df.drop(columns=["Portfolio"], errors="ignore")
            df = df.merge(
                new_assignments,
                on=["Ticker", "FormationYear", "Method"],
                how="inner"
            )
            df = df.rename(columns={"NewPortfolio": "Portfolio"})

    if weighting == "VW":
        valid = df["LagMarketCap"].notna() & df["Return"].notna() & (df["LagMarketCap"] > 0)
        df = df[valid].copy()
        df["WR"] = df["Return"] * df["LagMarketCap"]
        monthly = (
            df.groupby(["Date", "Variant", "Method", "Portfolio"])
            .apply(lambda g: g["WR"].sum() / g["LagMarketCap"].sum())
            .reset_index()
        )
    else:
        monthly = (
            df.groupby(["Date", "Variant", "Method", "Portfolio"])["Return"]
            .mean()
            .reset_index()
        )

    monthly.columns = ["Date", "Variant", "Method", "Portfolio", "Return"]
    monthly["Date"] = pd.to_datetime(monthly["Date"])
    return monthly

# ── EK filter setup ───────────────────────────────────────────────────────────
EK_FILTERS = {
    "No filter":   0.00,
    "EK/TK > 5%":  0.05,
    "EK/TK > 10%": 0.10,
    "EK/TK > 20%": 0.20,
}

def apply_ek_filter(holdings_df, threshold):
    if threshold == 0.0:
        return holdings_df
    vp = equity_ratio[equity_ratio["EK_ratio"] >= threshold][
        ["Ticker", "FormationYear"]
    ].drop_duplicates()
    return holdings_df.merge(vp, on=["Ticker", "FormationYear"], how="inner")

# ── Add Sector to holdings_all if not already present ────────────────────────
if "Sector" not in holdings_all.columns:
    sector_map = (
        step2[["Ticker", "Year", "Sector"]]
        .rename(columns={"Year": "FormationYear"})
        .assign(FormationYear=lambda x: x["FormationYear"] + 1)
        .drop_duplicates()
    )
    holdings_all = holdings_all.merge(
        sector_map, on=["Ticker", "FormationYear"], how="left"
    )
    n_missing = holdings_all["Sector"].isna().sum()
    n_total   = len(holdings_all)
    print(f"Sector added: {n_total - n_missing}/{n_total} rows covered "
          f"({(n_total - n_missing) / n_total * 100:.1f}%)")
else:
    print(f"Sector already present — skipping merge")

print(f"Unique sectors: {holdings_all['Sector'].nunique()}")

# ── Pre-compute all combinations ──────────────────────────────────────────────
holdings_analysis = holdings_all[holdings_all["Variant"] != "HB_down"].copy()

print("\nPre-computing all combinations...")
MONTHLY_RETURNS_EK = {}
total = len(["VW", "EW"]) * len([False, True]) * len(EK_FILTERS)
done  = 0

for w in ["VW", "EW"]:
    for n in [False, True]:
        for ek_label, ek_threshold in EK_FILTERS.items():
            filtered = apply_ek_filter(holdings_analysis, ek_threshold)
            MONTHLY_RETURNS_EK[(w, n, ek_label)] = compute_monthly_returns(
                filtered, w, n
            )
            done += 1
            print(f"  {done}/{total}  {w} | {'IN' if n else 'Std'} | {ek_label}")

print("\nAll combinations ready.")

# ── Shared widget helper ──────────────────────────────────────────────────────
def make_plot_widget(plot_fn):
    weight_dd  = widgets.Dropdown(
        options=["VW", "EW"],
        value="VW",
        description="Weighting:",
        layout=widgets.Layout(width="160px")
    )
    neutral_dd = widgets.Dropdown(
        options=[("Standard", False), ("Industry neutral", True)],
        value=False,
        description="Mode:",
        layout=widgets.Layout(width="210px")
    )
    ek_dd = widgets.Dropdown(
        options=list(EK_FILTERS.keys()),
        value="No filter",
        description="EK filter:",
        layout=widgets.Layout(width="200px")
    )
    output = widgets.Output()

    def update(change=None):
        ret = MONTHLY_RETURNS_EK[(weight_dd.value, neutral_dd.value, ek_dd.value)]
        label = (
            f"{weight_dd.value} — "
            f"{'Industry neutral' if neutral_dd.value else 'Standard'} — "
            f"{ek_dd.value}"
        )
        with output:
            output.clear_output(wait=True)
            plot_fn(ret, label)

    weight_dd.observe(update,  names="value")
    neutral_dd.observe(update, names="value")
    ek_dd.observe(update,      names="value")

    display(widgets.HBox([weight_dd, neutral_dd, ek_dd]))
    display(output)
    update()

In [ ]:
# ── Cell 12: Plot 1 — Cumulative return per quintile ─────────────────────────

QUINTILE_STYLES = {
    "Q1": {"color": PALETTE["red"],    "lw": 2.0},
    "Q2": {"color": PALETTE["orange"], "lw": 1.2},
    "Q3": {"color": PALETTE["green"],  "lw": 1.2},
    "Q4": {"color": PALETTE["blue"],   "lw": 1.2},
    "Q5": {"color": PALETTE["navy"],   "lw": 2.0},
}

def plot_cumulative_quintiles(ret, mode_label):
    variants = ["HB", "HB_cap"]

    fig, axes = plt.subplots(
        len(variants), len(LABELS),
        figsize=(18, 10),
        sharey=True, sharex=True
    )

    for row, variant in enumerate(variants):
        sub = ret[ret["Variant"] == variant]

        for col, (method, mlabel) in enumerate(LABELS.items()):
            ax = axes[row, col]

            for q, style in QUINTILE_STYLES.items():
                s = sub[
                    (sub["Method"] == method) &
                    (sub["Portfolio"] == q)
                ].set_index("Date").sort_index()["Return"]
                s = s[s.index.year >= START_YEAR].dropna()
                if len(s) == 0:
                    continue
                cum = (1 + s).cumprod() - 1
                ax.plot(cum.index, cum * 100,
                        color=style["color"],
                        linewidth=style["lw"],
                        alpha=0.9,
                        label=q)

            ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
            ax.set_title(f"{mlabel} — {variant}")
            ax.set_ylabel("Cumulative return (%)" if col == 0 else "")
            ax.yaxis.set_major_formatter(mtick.PercentFormatter())
            ax.legend(title="Quintile", fontsize=8)
            ax.grid(True, alpha=0.3)

    fig.suptitle(f"Cumulative return per quintile — {mode_label}", fontsize=13)
    plt.tight_layout()
    plt.show()

make_plot_widget(plot_cumulative_quintiles)

In [ ]:
# ── Cell 13: Plot 2 — Q5−Q1 spread ───────────────────────────────────────────

def plot_ls_spread(ret, mode_label):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

    for ax, variant in zip(axes, ["HB", "HB_cap"]):
        sub = ret[ret["Variant"] == variant]

        for method, mlabel in LABELS.items():
            q5 = sub[(sub["Method"] == method) & (sub["Portfolio"] == "Q5")] \
                 .set_index("Date")["Return"]
            q1 = sub[(sub["Method"] == method) & (sub["Portfolio"] == "Q1")] \
                 .set_index("Date")["Return"]
            common = q5.index.intersection(q1.index)
            ls  = q5.loc[common] - q1.loc[common]
            ls  = ls[ls.index.year >= START_YEAR].dropna()
            cum = (1 + ls).cumprod() - 1
            ax.plot(cum.index, cum * 100,
                    color=METHOD_COLORS[method],
                    linewidth=2, label=mlabel)

        ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
        ax.set_title(f"{variant} — {mode_label}")
        ax.set_ylabel("Cumulative return (%)")
        ax.yaxis.set_major_formatter(mtick.PercentFormatter())
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    fig.suptitle(f"Q5−Q1 L/S spread — {mode_label}", fontsize=13)
    plt.tight_layout()
    plt.show()

make_plot_widget(plot_ls_spread)

In [ ]:
# ── Cell 14: Plot 3 — Annualised return, vol and Sharpe ──────────────────────

RF_ANNUAL = 0.02

def plot_performance_bars(ret, mode_label):
    rows = []
    for variant in ["HB", "HB_cap"]:
        sub = ret[ret["Variant"] == variant]
        for method, mlabel in LABELS.items():
            for portfolio in ["Q1", "Q2", "Q3", "Q4", "Q5"]:
                s = sub[(sub["Method"] == method) & (sub["Portfolio"] == portfolio)] \
                    .set_index("Date")["Return"]
                s = s[s.index.year >= START_YEAR].dropna()
                if len(s) < 12:
                    continue
                ann_r = ((1 + s.mean()) ** 12 - 1) * 100
                ann_v = s.std() * np.sqrt(12) * 100
                rows.append({
                    "Variant":   variant,
                    "Method":    mlabel,
                    "Portfolio": portfolio,
                    "Ann. return (%)": ann_r,
                    "Ann. vol (%)":    ann_v,
                    "Sharpe":    (ann_r - RF_ANNUAL * 100) / ann_v,
                })

    df_perf = pd.DataFrame(rows)
    quintiles = ["Q1", "Q2", "Q3", "Q4", "Q5"]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=False)

    for row_idx, variant in enumerate(["HB", "HB_cap"]):
        sub = df_perf[df_perf["Variant"] == variant]

        for col_idx, metric in enumerate(["Ann. return (%)", "Ann. vol (%)", "Sharpe"]):
            ax = axes[row_idx, col_idx]
            x  = np.arange(len(quintiles))
            w  = 0.25

            for i, (method, mlabel) in enumerate(LABELS.items()):
                vals = sub[sub["Method"] == mlabel].set_index("Portfolio")[metric].reindex(quintiles)
                ax.bar(x + i * w, vals.values, w,
                       label=mlabel,
                       color=list(METHOD_COLORS.values())[i],
                       edgecolor="white", linewidth=0.8, alpha=0.85)

            ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
            ax.set_title(f"{metric} — {variant}")
            ax.set_xticks(x + w)
            ax.set_xticklabels(quintiles)
            ax.legend(fontsize=7)
            ax.grid(True, alpha=0.3, axis="y")

    fig.suptitle(f"Annualised performance by quintile — {mode_label}", fontsize=13)
    plt.tight_layout()
    plt.show()

make_plot_widget(plot_performance_bars)

In [ ]:
# ── Cell 16: Plot 5 — Rolling Sharpe ─────────────────────────────────────────

def plot_rolling_sharpe(ret, mode_label):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

    roll_dd = widgets.Dropdown(
        options=[("12 months", 12), ("24 months", 24)],
        value=12,
        description="Window:",
        layout=widgets.Layout(width="180px")
    )

    for ax, variant in zip(axes, ["HB", "HB_cap"]):
        sub = ret[ret["Variant"] == variant]

        for method, mlabel in LABELS.items():
            q5 = sub[(sub["Method"] == method) & (sub["Portfolio"] == "Q5")] \
                 .set_index("Date")["Return"]
            q1 = sub[(sub["Method"] == method) & (sub["Portfolio"] == "Q1")] \
                 .set_index("Date")["Return"]
            common = q5.index.intersection(q1.index)
            ls_ret = (q5.loc[common] - q1.loc[common])
            ls_ret = ls_ret[ls_ret.index.year >= START_YEAR].dropna()

            window = 12
            rolling_sharpe = (
                ls_ret.rolling(window).mean() /
                ls_ret.rolling(window).std()
            ) * np.sqrt(12)

            ax.plot(rolling_sharpe.index, rolling_sharpe.values,
                    color=METHOD_COLORS[method],
                    linewidth=2, label=mlabel)

        ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
        ax.set_title(f"{variant} — {mode_label}")
        ax.set_ylabel("Rolling Sharpe (12m)" if ax == axes[0] else "")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    fig.suptitle(f"Rolling 12-month Sharpe — Q5−Q1 spread — {mode_label}",
                 fontsize=13)
    plt.tight_layout()
    plt.show()

make_plot_widget(plot_rolling_sharpe)

In [ ]:
# ── Cell 11c: Factor model setup ──────────────────────────────────────────────
from scipy import stats as scipy_stats
import statsmodels.api as sm

FACTORS_CSV = ROOT / "results" / "extraction_static" / "factor_data.csv"
factors = pd.read_csv(FACTORS_CSV, parse_dates=["Date"])
factors["Date"] = pd.to_datetime(factors["Date"])
factors = factors.set_index("Date")

FACTOR_MODELS = {
    "CAPM":        ["MKT"],
    "FF3":         ["MKT", "SMB", "HML"],
    "FF3+MOM":     ["MKT", "SMB", "HML", "MOM"],
    "FF5":         ["MKT", "SMB", "HML", "RMW", "CMA"],
    "FF5+MOM":     ["MKT", "SMB", "HML", "RMW", "CMA", "MOM"],
}

def run_factor_regression(ret_series, factor_cols, rf_series, nw_lags=12):
    """
    Runs OLS regression of excess returns on factors with Newey-West SEs.
    Returns dict with alpha, t_stat, p_value, r_squared and factor betas.
    """
    df = pd.DataFrame({
        "ret": ret_series,
        "rf":  rf_series,
    }).join(factors[factor_cols]).dropna()

    if len(df) < 24:
        return None

    df["excess_ret"] = df["ret"] - df["rf"]
    X = sm.add_constant(df[factor_cols])
    y = df["excess_ret"]

    model  = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": nw_lags})

    result = {
        "alpha":     model.params["const"] * 12,        # annualised
        "t_stat":    model.tvalues["const"],
        "p_value":   model.pvalues["const"],
        "r_squared": model.rsquared,
        "n_obs":     int(len(df)),
    }
    for f in factor_cols:
        result[f"beta_{f}"] = model.params[f]

    return result


def run_all_factor_regressions(ret_df, nw_lags=12):
    """
    Runs all five factor models for all variants, methods and portfolios.
    Returns a long-format DataFrame.
    """
    rf = factors["RF"]
    rows = []

    for variant in ["HB", "HB_cap"]:
        sub_v = ret_df[ret_df["Variant"] == variant]

        for method, mlabel in LABELS.items():
            sub_m = sub_v[sub_v["Method"] == method]

            for portfolio in ["Q1", "Q2", "Q3", "Q4", "Q5"]:
                s = sub_m[sub_m["Portfolio"] == portfolio] \
                    .set_index("Date")["Return"]
                s = s[s.index.year >= START_YEAR].dropna()

                for model_name, factor_cols in FACTOR_MODELS.items():
                    res = run_factor_regression(s, factor_cols, rf, nw_lags)
                    if res is None:
                        continue
                    rows.append({
                        "Variant":   variant,
                        "Method":    mlabel,
                        "Portfolio": portfolio,
                        "Model":     model_name,
                        **res,
                    })

            # Q5-Q1 spread
            q5 = sub_m[sub_m["Portfolio"] == "Q5"].set_index("Date")["Return"]
            q1 = sub_m[sub_m["Portfolio"] == "Q1"].set_index("Date")["Return"]
            common = q5.index.intersection(q1.index)
            ls = (q5.loc[common] - q1.loc[common])
            ls = ls[ls.index.year >= START_YEAR].dropna()

            for model_name, factor_cols in FACTOR_MODELS.items():
                res = run_factor_regression(ls, factor_cols, rf, nw_lags)
                if res is None:
                    continue
                rows.append({
                    "Variant":   variant,
                    "Method":    mlabel,
                    "Portfolio": "Q5-Q1",
                    "Model":     model_name,
                    **res,
                })

    return pd.DataFrame(rows)


print("Factor model setup ready.")
print(f"Models: {list(FACTOR_MODELS.keys())}")
print(f"Factor data: {factors.index.min()} → {factors.index.max()}")

In [ ]:
# ── Cell 17: Factor model plots ───────────────────────────────────────────────

def plot_factor_models(ret, mode_label):
    print(f"Running factor regressions — {mode_label}...")
    results = run_all_factor_regressions(ret)

    # ── Plot 1: Alpha heatmap — Q5-Q1 across models and methods ──────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    for ax, variant in zip(axes, ["HB", "HB_cap"]):
        sub = results[
            (results["Variant"]   == variant) &
            (results["Portfolio"] == "Q5-Q1")
        ].copy()

        pivot_alpha = sub.pivot(index="Model", columns="Method", values="alpha")
        pivot_tstat = sub.pivot(index="Model", columns="Method", values="t_stat")

        pivot_alpha = pivot_alpha.reindex(list(FACTOR_MODELS.keys()))
        pivot_tstat = pivot_tstat.reindex(list(FACTOR_MODELS.keys()))

        im = ax.imshow(pivot_alpha.values, cmap="RdYlGn", aspect="auto",
                       vmin=-0.10, vmax=0.10)

        ax.set_xticks(range(len(pivot_alpha.columns)))
        ax.set_xticklabels(pivot_alpha.columns, rotation=20, ha="right", fontsize=9)
        ax.set_yticks(range(len(pivot_alpha.index)))
        ax.set_yticklabels(pivot_alpha.index, fontsize=9)
        ax.set_title(f"Annualised alpha (Q5−Q1) — {variant}")

        for i in range(len(pivot_alpha.index)):
            for j in range(len(pivot_alpha.columns)):
                a = pivot_alpha.values[i, j]
                t = pivot_tstat.values[i, j]
                stars = "***" if abs(t) > 3.09 else "**" if abs(t) > 2.33 else "*" if abs(t) > 1.65 else ""
                ax.text(j, i, f"{a*100:.1f}%{stars}\n({t:.2f})",
                        ha="center", va="center", fontsize=8,
                        color="black")

        plt.colorbar(im, ax=ax, label="Alpha (annualised)")

    fig.suptitle(f"Risk-adjusted alpha — {mode_label}", fontsize=13)
    plt.tight_layout()
    plt.show()

    # ── Plot 2: Alpha per quintile — FF5+MOM ─────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
    quintiles = ["Q1", "Q2", "Q3", "Q4", "Q5"]
    x = np.arange(len(quintiles))
    w = 0.25

    for ax, variant in zip(axes, ["HB", "HB_cap"]):
        sub = results[
            (results["Variant"] == variant) &
            (results["Model"]   == "FF5+MOM") &
            (results["Portfolio"].isin(quintiles))
        ]

        for i, (method, mlabel) in enumerate(LABELS.items()):
            vals = sub[sub["Method"] == mlabel] \
                   .set_index("Portfolio")["alpha"].reindex(quintiles) * 100
            ax.bar(x + i * w, vals.values, w,
                   label=mlabel,
                   color=list(METHOD_COLORS.values())[i],
                   edgecolor="white", linewidth=0.8, alpha=0.85)

        ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
        ax.set_title(f"FF5+MOM alpha by quintile — {variant}")
        ax.set_xticks(x + w)
        ax.set_xticklabels(quintiles)
        ax.set_ylabel("Annualised alpha (%)")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, axis="y")

    fig.suptitle(f"FF5+MOM alpha by quintile — {mode_label}", fontsize=13)
    plt.tight_layout()
    plt.show()

    # ── Plot 3: Factor loadings for Q5 — FF5+MOM ─────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
    factor_cols = ["MKT", "SMB", "HML", "RMW", "CMA", "MOM"]
    x = np.arange(len(factor_cols))

    for ax, variant in zip(axes, ["HB", "HB_cap"]):
        sub = results[
            (results["Variant"]   == variant) &
            (results["Model"]     == "FF5+MOM") &
            (results["Portfolio"] == "Q5")
        ]

        for i, (method, mlabel) in enumerate(LABELS.items()):
            row  = sub[sub["Method"] == mlabel]
            vals = [row[f"beta_{f}"].values[0] if len(row) > 0 else np.nan
                    for f in factor_cols]
            ax.bar(x + i * w, vals, w,
                   label=mlabel,
                   color=list(METHOD_COLORS.values())[i],
                   edgecolor="white", linewidth=0.8, alpha=0.85)

        ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
        ax.set_title(f"Factor loadings Q5 (FF5+MOM) — {variant}")
        ax.set_xticks(x + w)
        ax.set_xticklabels(factor_cols)
        ax.set_ylabel("Beta")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, axis="y")

    fig.suptitle(f"Q5 factor loadings — {mode_label}", fontsize=13)
    plt.tight_layout()
    plt.show()

    # ── Interactive stats table ───────────────────────────────────────────────
    model_dd = widgets.Dropdown(
        options=list(FACTOR_MODELS.keys()),
        value="FF5+MOM",
        description="Model:",
        layout=widgets.Layout(width="180px")
    )
    port_dd = widgets.Dropdown(
        options=["Q5-Q1", "Q1", "Q2", "Q3", "Q4", "Q5"],
        value="Q5-Q1",
        description="Portfolio:",
        layout=widgets.Layout(width="180px")
    )
    tbl_output = widgets.Output()

    def update_table(change=None):
        sub = results[
            (results["Model"]     == model_dd.value) &
            (results["Portfolio"] == port_dd.value)
        ][["Variant", "Method", "alpha", "t_stat", "p_value", "r_squared", "n_obs"]].copy()
        sub["alpha"] = (sub["alpha"] * 100).round(2)
        sub["t_stat"]    = sub["t_stat"].round(2)
        sub["p_value"]   = sub["p_value"].round(3)
        sub["r_squared"] = sub["r_squared"].round(3)

        def color_alpha(val):
            return "color: green" if val > 0 else "color: red"

        styled = (
            sub.style
            .hide(axis="index")
            .map(color_alpha, subset=["alpha"])
            .format({"alpha": "{:.2f}%", "t_stat": "{:.2f}",
                     "p_value": "{:.3f}", "r_squared": "{:.3f}"})
        )
        with tbl_output:
            tbl_output.clear_output(wait=True)
            display(styled)

    model_dd.observe(update_table, names="value")
    port_dd.observe(update_table,  names="value")
    display(widgets.HBox([model_dd, port_dd]))
    display(tbl_output)
    update_table()

make_plot_widget(plot_factor_models)